# 2. Data Preprocessing

## Purpose
Transform raw data into a clean, numeric feature matrix ready for modelling.
Fits the preprocessing pipeline on training data only to prevent data leakage,
then applies it to both train and test sets.

## Inputs
- `data/raw/` — raw dataset (CSV)
- `config.yaml` — target column, columns to drop, split parameters, pipeline settings

## Outputs
- `data/processed/X_train.csv` — preprocessed training features
- `data/processed/X_test.csv` — preprocessed test features
- `data/processed/y_train.csv` — training labels
- `data/processed/y_test.csv` — test labels
- `artifacts/preprocessor.pkl` — fitted preprocessing pipeline

## Decisions for the user
- Set `target` in `config.yaml` — the column you are predicting
- Set `drop_columns` in `config.yaml` — ID columns or any columns to exclude
- Set `split.test_size` in `config.yaml` — typically 0.2 for most projects
- Set `split.stratify` in `config.yaml` — true for classification, false for regression
- Set `pipeline.scaler` in `config.yaml` — standard, minmax or robust
- Review the sparsity output — if density is very low consider dropping
  high missingness columns via `drop_columns`

## Important note
The test set is held out here and not used again until notebook 7.
All modelling decisions are made using cross validation scores only.

In [1]:
#Import required libraries and project API functions

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from src.data.load_data import load_raw_data
from src.config.settings import load_config
from src.config.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from src.features.pipeline import FeatureEngineeringPipeline
from src.features.type_detection import detect_feature_types


## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
PIPELINE_CONFIG = config["pipeline"]

print(f"Target:        {TARGET}")
print(f"Dropping:      {DROP_COLS}")
print(f"Test size:     {SPLIT_CONFIG['test_size']}")
print(f"Stratify:      {SPLIT_CONFIG['stratify']}")         

Target:        good
Dropping:      ['quality']
Test size:     0.2
Stratify:      True


## 2. Load data

In [3]:
df = load_raw_data(na_values=config.get("na_values", []))
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,good
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


## 3. Split features and target

In [4]:
from src.data.load_data import prepare_data



X, y = prepare_data(df, config)

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

240 instances have been dropped
Features shape: (1359, 11)
Target distribution:
good
1    719
0    640
Name: count, dtype: int64


## 4. Train/test split
The test set is held out completely until final evaluation.
Stratification preserves class balance between train and test sets.

In [5]:
from sklearn.model_selection import train_test_split

stratify_col = y if SPLIT_CONFIG["stratify"] else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nTarget distribution in train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTarget distribution in test:\n{y_test.value_counts(normalize=True).round(3)}")
     

Train size: (1087, 11)
Test size:  (272, 11)

Target distribution in train:
good
1    0.529
0    0.471
Name: proportion, dtype: float64

Target distribution in test:
good
1    0.529
0    0.471
Name: proportion, dtype: float64


## 5. Feature type detection
Automatically detects numeric, binary, categorical, high cardinality
and datetime features. Review the output before fitting the pipeline.

In [6]:
feature_types = detect_feature_types(X_train)
feature_types


{'numeric': ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol'],
 'binary': [],
 'categorical': [],
 'high_cardinality': [],
 'datetime': [],
 'ordinal': []}

## 6. Configure and fit pipeline
The pipeline is fitted on training data only. It will be applied to
both train and test sets in the next step.

In [7]:
preprocessor = FeatureEngineeringPipeline(config=PIPELINE_CONFIG)
preprocessor.fit(X_train, target=y_train)


## 7. Transform train and test
The same fitted pipeline is applied to both sets ensuring identical
transformations without any information leakage from the test set.

In [8]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape:  {X_test_transformed.shape}")

X_train_transformed shape: (1087, 11)
X_test_transformed shape:  (272, 11)


## 8. Check transformed features
Verify the transformed data has no NaN values, no infinite values
and reasonable sparsity before saving.

In [9]:
X_train_transformed.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
973,-0.804643,-0.439848,-0.827031,-0.104893,-0.890990,-0.947526,-1.022071,-1.695064,-0.076882,-0.051041,0.882388
890,-0.804643,-0.494133,-1.384997,-0.823349,-0.360431,1.560222,-0.257078,-1.246799,0.053493,-0.454755,0.882388
855,-0.060652,-0.548418,0.086002,-0.679658,-0.144277,1.077963,-0.051118,-0.366278,-0.402820,-0.685449,-0.130951
1351,-1.148023,0.103002,-0.624135,-0.535967,-0.222879,0.981511,-0.345346,-1.908523,0.053493,0.929407,1.066631
801,-1.205253,-0.385563,-0.522687,-0.679658,-0.301480,-0.851075,-1.051494,-1.316173,1.943934,-0.685449,0.882388


In [10]:
#Check for NA's

train_nas = X_train_transformed.isna().sum().sum()
test_nas = X_test_transformed.isna().sum().sum()
print(f"NAs in train: {train_nas}")
print(f"NAs in test:  {test_nas}")  

NAs in train: 0
NAs in test:  0


In [11]:
#Check for infinite values

train_inf = np.isinf(X_train_transformed.to_numpy()).sum()
test_inf = np.isinf(X_test_transformed.to_numpy()).sum()
print(f"Infinite values in train: {train_inf}")
print(f"Infinite values in test:  {test_inf}")

Infinite values in train: 0
Infinite values in test:  0


In [12]:
#Check sparsity

density = (X_train_transformed != 0).sum().sum() / X_train_transformed.size
print(f"Train density: {density:.3f}")

# If density is below 0.1, consider:
# - Dropping high missingness columns via drop_columns in config.yaml
# - Reducing hashing_dim in config.yaml for high cardinality features
# - Extracting meaningful features before preprocessing (feature engineering notebook)


Train density: 1.000


## 9. Save processed data and preprocessor
Processed data is saved to `data/processed/` for use by all downstream
notebooks. The fitted preprocessor is saved to `artifacts/` so it can
be applied consistently to new data at prediction time.

In [13]:
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

    X_train_transformed.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
    X_test_transformed.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)
    y_train.to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
    y_test.to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)

    print(f"Processed data saved to {PROCESSED_DATA_DIR}")

    # %%
    # 13. Save Preprocessor
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.pkl")
    print(f"Preprocessor saved to {ARTIFACTS_DIR / 'preprocessor.pkl'}")

Processed data saved to /home/kigr/wine-quality/data/processed
Preprocessor saved to /home/kigr/wine-quality/artifacts/preprocessor.pkl
